In [ ]:
#| default_exp quantize.fake_quantizer

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import torch
import torch.nn as nn
from einops import rearrange
from fastcore.basics import store_attr
from fasterai.core.precision import (FAKE_SPEC_ATTR, FakeQuantSpec, NEEDS_GROUP_SIZE, QSCHEMES,
                                     _check_group_size, _check_qscheme, _type_error)

## Overview

`FakeQuantizer` rounds a model's weights and activations onto the grids of the widths you ask for, and
leaves everything in floating point. Nothing is packed, no kernel is swapped, no backend is involved: the
model still runs its ordinary `nn.Conv2d` and `nn.Linear` modules, at its original dtype, on the device it
was trained on. What it answers is one question — *what does this width cost in accuracy?* — before
spending a deployment flow to find out.

That is what makes any pair askable. `W4A8`, `W2A8` and `W8A4` are all arithmetic here, where a backend
table only lists the pairs somebody shipped a kernel for. When you have chosen a width and want a model
that actually runs at it, that is [Quantizer](quantizer.html)'s job.

A few things this engine deliberately does not do:

- **Biases stay in floating point.** A deployed integer kernel accumulates them at int32; here they are
  untouched, so a width claim from this engine is about the weights and activations only.
- **BatchNorm is not folded.** A deployed flow folds it before quantizing, which changes the per-channel
  weight ranges; fold it yourself first with [BN_Folder](../misc/bn_folding.html) if you want the ranges
  a folded graph would see.
- **`nn.Embedding` and `nn.ConvTranspose2d` are outside the default `layer_type`.** A model holding them
  keeps those tensors byte-identical while `print_precision()` still names a width, so a width reported on
  a transformer does not cover its largest tensor unless you name that type yourself.

## The arithmetic

The rounding wraps `torch.fake_quantize_per_tensor_affine` and `torch.fake_quantize_per_channel_affine`,
which accept any width from 2 to 16 bits on CPU and on CUDA. `per_group` reshapes each row into groups
with `einops` and reuses the per-channel operator, refusing a `group_size` that does not divide the row.

Torch's own `MinMaxObserver._calculate_qparams` is not reused for the same reason a backend table cannot
express `W4A8`: an observer is bound to a real quantized dtype, so widths such as 2, 6 or 13 have no
`torch.qint*` to be calculated against. Computing the scale here is what makes every width sayable.

A grid always contains 0: the range is widened to include it, and the scale is clamped to an epsilon so a
zero-range channel cannot produce a zero scale. One consequence worth knowing — a weight that is exactly
0 quantizes to the zero-point and dequantizes to exactly 0, so sparsity survives the rounding, symmetric
or affine.

In [ ]:
#| export
_EPS = torch.finfo(torch.float32).eps  # a zero range would otherwise give a zero scale
_OBSERVERS = ('static', 'dynamic')


def _check_bits(name: str, value, tail: str = 'or None to stay in floating point'):
    "Validate one bit width; None means the tensor stays in floating point"
    if value is None: return None
    if isinstance(value, bool) or not isinstance(value, int):
        raise _type_error(name, f'an int in [2, 16], {tail}', value)
    if not 2 <= value <= 16:
        raise ValueError(f"`{name}={value}` is not a width this engine rounds to: pass an int in [2, 16], "
                         f"{tail}.")
    return value


def _group_refusal(group_size: int, row_len: int, where: str) -> ValueError:
    "One sentence for a group that does not divide a row, wherever it is caught"
    return ValueError(f"group_size={group_size} does not divide the {row_len} weights of a row of {where}: "
                      "pass a group_size that divides it, or qscheme='per_channel'.")


def _qrange(bits: int, symmetric: bool) -> tuple[int, int]:
    "Lowest and highest integer of a `bits`-wide grid"
    if symmetric: return -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    return 0, 2 ** bits - 1


def _scale_zero(lo, hi, bits: int, symmetric: bool):
    "Scale and integer zero-point of the grid covering [lo, hi], which always contains 0"
    qmin, qmax = _qrange(bits, symmetric)
    lo, hi = lo.clamp(max=0), hi.clamp(min=0)
    if symmetric:
        scale = torch.maximum(hi, -lo) / qmax
        zero = torch.zeros_like(scale)
    else:
        scale = ((hi - lo) / (qmax - qmin)).clamp(min=_EPS)  # a zero scale would give a nan zero-point
        zero = qmin - torch.round(lo / scale)
    return scale.clamp(min=_EPS).float(), zero.clamp(qmin, qmax).to(torch.int32)


def _qparams(t: torch.Tensor, bits: int, symmetric: bool, per_row: bool = False):
    "Scale and zero-point of `t`: one pair for the whole tensor, or one per row"
    lo, hi = (t.amin(1), t.amax(1)) if per_row else (t.amin(), t.amax())
    return _scale_zero(lo, hi, bits, symmetric)


def _fake_quantize(t, bits, symmetric, qscheme='per_tensor', group_size=None):
    "Round `t` onto a `bits`-wide grid, keeping its shape and floating-point dtype"
    qmin, qmax = _qrange(bits, symmetric)
    if qscheme == 'per_tensor':
        scale, zero = _qparams(t, bits, symmetric)
        return torch.fake_quantize_per_tensor_affine(t, scale, zero, qmin, qmax)
    rows = t.flatten(1)
    if qscheme == 'per_group':
        if rows.shape[1] % group_size: raise _group_refusal(group_size, rows.shape[1], 'this tensor')
        rows = rearrange(rows, 'o (g s) -> (o g) s', s=group_size)
    scale, zero = _qparams(rows, bits, symmetric, per_row=True)
    return torch.fake_quantize_per_channel_affine(rows, scale, zero, 0, qmin, qmax).reshape(t.shape)

Calibration reads the fastai interface and nothing else: a `DataLoaders`, or one of the loaders it
holds. A torch `DataLoader`, a list of batches or a bare tensor is refused by name, pointing at `dls` or
`dls.train` — fasterai trains and validates through fastai, and calibration is a validation pass.

Only a `DataLoaders` is unwrapped to its training loader, and it is recognised by carrying `loaders`,
not by answering to `.train`: a fastai loader delegates an unknown `.train` to its dataset, which yields
*unbatched items*, so `hasattr(dl, 'train')` is `True` and reading it would calibrate on the wrong shapes.
Narrowing the accepted types does not remove that trap, because it lives inside fastai.

The activation scales live as buffers on the module that produces them, and the hook that rounds them,
`_ActRounder(bits, symmetric, dynamic)`, is a picklable class rather than a closure, because that hook stays installed — `copy.deepcopy` treats a closure as
atomic, so a copy would share the original's calibration. The observation hooks below may be closures:
they are removed in a `finally` before anyone can copy the model.

In [ ]:
#| export
def _batch_input(batch) -> torch.Tensor:
    "Input part of a calibration batch, whether it is a tensor or an (input, target) tuple"
    x = batch[0] if isinstance(batch, (list, tuple)) else batch
    if not isinstance(x, torch.Tensor):
        raise TypeError(f"Calibration data must yield tensors, got {type(x).__name__}.")
    return x.as_subclass(torch.Tensor)


def _calib_inputs(data, n_batches: int):
    "Input tensors from a fastai `DataLoaders` or one of its loaders"
    if not hasattr(data, 'one_batch'):
        raise TypeError(f"Calibration data must be a fastai `DataLoaders` or one of its loaders, got "
                        f"{type(data).__name__}: pass `dls` or `dls.train`.")
    # only a `DataLoaders` carries `loaders`; a fastai loader delegates an unknown `.train` to its
    # dataset, which yields unbatched items — reading `.train` off one is the trap this line exists
    # to avoid, and narrowing the accepted types does not remove it
    loader = data.train if hasattr(data, 'loaders') and hasattr(data, 'train') else data
    for n, batch in enumerate(loader):
        if n >= n_batches: break
        yield _batch_input(batch)


def _width_label(bits) -> str:
    "How a width reads in a report"
    return 'float' if bits is None else f'{bits} bits'


class _ActRounder:
    "Forward hook rounding a module's output onto its activation grid"
    def __init__(self, bits: int, symmetric: bool, dynamic: bool):
        store_attr()
        self.enabled = True

    def __call__(self, mod, inp, out):
        if not self.enabled or not isinstance(out, torch.Tensor): return out
        if self.dynamic:
            scale, zero = _qparams(out.detach(), self.bits, self.symmetric)  # the op refuses a scale with grad
        else:
            scale, zero = getattr(mod, '_act_scale', None), getattr(mod, '_act_zero_point', None)
            if scale is None:
                raise RuntimeError("Activation scales are missing: call calibrate(data) before "
                                   "quantize_model(), or pass observer='dynamic'.")
        return torch.fake_quantize_per_tensor_affine(out, scale, zero, *_qrange(self.bits, self.symmetric))

## Provenance

The widths a quantizer was asked for are available as `fq.spec`, and are attached to every model it
rounds: [`fake_quant_spec(model)`](../core/precision.html) reads them back. `FakeQuantSpec` lives beside
`QuantSpec` in [Precision](../core/precision.html), so asking "what precision is this model at?" needs no
quantizer import — that page also lists the three places where its vocabulary deliberately diverges from
the deployed one. It records what was *asked for*; it says nothing about what a runtime would do with it.

---

## FakeQuantizer

In [ ]:
#| export
class FakeQuantizer:
    """Simulated quantization: rounds weights and activations onto the widths asked for, in floating point.

    The model keeps its floating-point dtype and its ordinary modules, so this measures what a width
    costs in accuracy, not what it saves. Biases stay in floating point, where an integer kernel would
    carry them at int32. `nn.Embedding` and `nn.ConvTranspose2d` are outside the default `layer_type`,
    so a model holding them keeps those tensors byte-identical while the report still names a width."""

    def __init__(self,
                 model: nn.Module,
                 weight_bits: int | None = 8,          # Weight width; None leaves them in floating point
                 act_bits: int | None = None,          # Activation width; None leaves them in floating point
                 qscheme: str = 'per_channel',         # Weight scale axis: 'per_tensor', 'per_channel' or 'per_group'
                 symmetric: bool = True,               # Zero-point 0, or an affine offset
                 group_size: int | None = None,        # Weights sharing one scale (qscheme='per_group')
                 observer: str = 'static',             # Activation observer: 'static' (scales frozen by `calibrate`) or 'dynamic'
                 layer_type: type | tuple = (nn.Conv2d, nn.Linear),  # Module types that carry the rounding
                 layer_bits: dict | None = None,       # {layer_name: width} overriding `weight_bits`
                 layer_act_bits: dict | None = None,   # {layer_name: width} overriding `act_bits`
    ):
        weight_bits = _check_bits('weight_bits', weight_bits)
        act_bits = _check_bits('act_bits', act_bits)
        qscheme = _check_qscheme(qscheme)
        if observer not in _OBSERVERS:
            raise ValueError(f"Unknown observer '{observer}'. Pass 'static' (scales frozen by calibrate) or "
                             "'dynamic' (scales recomputed on every batch).")
        if qscheme == 'per_group':
            if group_size is None: raise ValueError(NEEDS_GROUP_SIZE.format(example=64))
            _check_group_size(group_size)
        elif group_size is not None:
            raise ValueError(f"`group_size={group_size}` only means something with qscheme='per_group'; "
                             f"this quantizer rounds weights '{qscheme}'.")
        layer_type = layer_type if isinstance(layer_type, tuple) else (layer_type,)
        store_attr()
        self._act_hooks, self._calibrated, self._quantized = {}, False, False
        self._named = list(self._iter_named_layers())
        self._weight_map = self._to_bits_dict(weight_bits, layer_bits, 'layer_bits')
        self._act_map = self._to_bits_dict(act_bits, layer_act_bits, 'layer_act_bits')
        self._check_model()

    def _iter_named_layers(self):
        "Iterate over the (name, module) pairs this quantizer rounds"
        for name, m in self.model.named_modules():
            if isinstance(m, self.layer_type): yield name, m

    def _to_bits_dict(self, default, overrides: dict | None, arg_name: str) -> dict:
        "Map every selected module to its width, the per-layer dict overriding the default"
        bits = {m: default for _, m in self._named}
        if not overrides: return bits
        by_name = dict(self._named)
        for key, b in overrides.items():
            label = key if isinstance(key, str) else type(key).__name__
            m = by_name.get(key) if isinstance(key, str) else key
            if m in bits: bits[m] = _check_bits(f"{arg_name}['{label}']", b)
            else: print(f"Warning: Layer '{label}' not found among the layers this quantizer rounds, skipping")
        return bits

    def _check_model(self) -> None:
        "Refuse a model this grammar cannot round, before a single weight is written"
        if not self._named:
            raise ValueError(f"No {'/'.join(t.__name__ for t in self.layer_type)} in this model: pass "
                             "layer_type= the module types that carry the weights to round.")
        if self.qscheme == 'per_tensor': return
        for name, m in self._named:
            if self._weight_map[m] is None: continue
            if isinstance(m, nn.modules.conv._ConvTransposeNd):
                raise ValueError(f"'{name}' is a {type(m).__name__}, whose weight is [in, out/groups, "
                                 "*kernel], so dim 0 is its input axis: round it with qscheme='per_tensor'.")
            row = m.weight.flatten(1).shape[1]
            if self.qscheme == 'per_group' and row % self.group_size:
                raise _group_refusal(self.group_size, row, f"'{name}'")

    @property
    def spec(self) -> FakeQuantSpec:
        "The widths this quantizer was asked for"
        return FakeQuantSpec(weight_bits=self.weight_bits, act_bits=self.act_bits, qscheme=self.qscheme,
                             symmetric=self.symmetric, observer=self.observer, group_size=self.group_size,
                             layer_bits=dict(self.layer_bits) if self.layer_bits else None,
                             layer_act_bits=dict(self.layer_act_bits) if self.layer_act_bits else None)

    def calibrate(self,
                  data,                 # A fastai `DataLoaders`, or one of its loaders
                  n_batches: int = 5,   # Batches to observe
    ) -> None:
        "Observe the activation ranges on `data` and freeze the static scales"
        sites = {m: b for m, b in self._act_map.items() if b is not None}
        if not sites:
            raise ValueError("There is no activation to calibrate: act_bits=None leaves the activations in "
                             "floating point. Pass act_bits=8, or a layer_act_bits dict.")
        device, ranges, seen = next(self.model.parameters()).device, {}, 0

        def make_hook(mod):
            def hook(_, inp, out):  # a closure is safe here: these handles come off in the `finally`
                if not isinstance(out, torch.Tensor): return
                lo, hi = out.detach().amin(), out.detach().amax()
                if mod in ranges:
                    lo, hi = torch.minimum(ranges[mod][0], lo), torch.maximum(ranges[mod][1], hi)
                ranges[mod] = (lo, hi)
            return hook

        handles = [m.register_forward_hook(make_hook(m)) for m in sites]
        for _, rounder in self._act_hooks.values(): rounder.enabled = False  # observe, do not round
        was_training = self.model.training
        self.model.eval()
        try:
            with torch.no_grad():
                for x in _calib_inputs(data, n_batches):
                    self.model(x.to(device))
                    seen += 1
        finally:
            for h in handles: h.remove()
            for _, rounder in self._act_hooks.values(): rounder.enabled = True
            self.model.train(was_training)
        if not seen:
            raise ValueError("calibrate() saw no batch: pass data holding at least one batch.")
        for name, m in self._named:
            if m in sites and m not in ranges:
                raise ValueError(f"'{name}' emitted no tensor over {seen} batch(es), so nothing measured its "
                                 "activation range: leave it in floating point with layer_act_bits, or pass "
                                 "data that reaches it.")
        for m, bits in sites.items():
            scale, zero = _scale_zero(*ranges[m], bits, self.symmetric)
            m.register_buffer('_act_scale', scale, persistent=False)
            m.register_buffer('_act_zero_point', zero, persistent=False)
        self._calibrated = True

    def quantize_model(self) -> nn.Module:
        "Round the weights and hook the activations; the model is changed in place and returned"
        if self._quantized:
            raise RuntimeError("This model is already rounded: call remove() before rounding it again.")
        if self.observer == 'static' and not self._calibrated \
                and any(b is not None for b in self._act_map.values()):
            raise ValueError("observer='static' rounds activations onto frozen scales that nothing has "
                             "measured yet: call calibrate(data) before quantize_model(), or pass "
                             "observer='dynamic'.")
        for _, m in self._named:
            bits = self._weight_map[m]
            if bits is not None:
                # a retry after a mid-loop failure must keep the first snapshot, not the rounded weight
                if '_fp_weight' not in m._buffers:
                    m.register_buffer('_fp_weight', m.weight.detach().clone(), persistent=False)
                m.weight.data.copy_(_fake_quantize(m.weight.data, bits, self.symmetric,
                                                   self.qscheme, self.group_size))
            act_bits = self._act_map[m]
            if act_bits is not None and m not in self._act_hooks:
                rounder = _ActRounder(act_bits, self.symmetric, self.observer == 'dynamic')
                self._act_hooks[m] = (m.register_forward_hook(rounder), rounder)
        setattr(self.model, FAKE_SPEC_ATTR, self.spec)
        self._quantized = True
        return self.model

    def remove(self) -> nn.Module:
        "Restore the floating-point weights and drop every hook and buffer; returns the same model"
        for handle, _ in self._act_hooks.values(): handle.remove()
        self._act_hooks = {}
        for _, m in self._named:
            if '_fp_weight' in m._buffers: m.weight.data.copy_(m._fp_weight)
            for name in ('_fp_weight', '_act_scale', '_act_zero_point'):
                if name in m._buffers: delattr(m, name)
        if hasattr(self.model, FAKE_SPEC_ATTR): delattr(self.model, FAKE_SPEC_ATTR)
        self._quantized, self._calibrated = False, False
        return self.model

    def print_precision(self) -> None:
        "Print the width every layer is rounded to"
        print("\nSimulated Precision Report:")
        print("-" * 80)
        print(f"{'Layer':<32} {'Type':<14} {'Weight':<10} {'Act':<10} {'Weight axis':<12}")
        print("-" * 80)
        for name, m in self._named:
            w, a = self._weight_map[m], self._act_map[m]
            axis = self.qscheme if w is not None else '-'
            print(f"{name:<32} {type(m).__name__:<14} {_width_label(w):<10} {_width_label(a):<10} {axis:<12}")
        print("-" * 80)
        print(f"{'Overall':<32} {self.spec.label:<14}")
        print("Rounded in floating point: every tensor is still stored at its original width, and the model "
              "holds a floating-point copy of every weight it rounds until remove().")

In [ ]:
show_doc(FakeQuantizer)

The arguments name the four things a rounding grid needs:

- `weight_bits` / `act_bits`: the widths. `None` leaves that tensor in floating point — a `16` is a real
  16-bit grid, not a way of saying "float".
- `qscheme`: the axis the weight scales are computed along, `'per_tensor'`, `'per_channel'` or
  `'per_group'`. This is the scale axis, not the [granularity](../core/granularity.html) a `Sparsifier`
  removes along. Activations are always rounded per tensor.
- `symmetric`: a zero-point of 0, or an affine offset.
- `observer`: `'static'` freezes the activation scales measured by `calibrate`, `'dynamic'` recomputes
  them on every batch. Unlike [Quantizer](quantizer.html)'s `method=`, which names a whole flow, this
  names only how the activation scales are obtained.

Per-layer widths follow the siblings' form — a dict against the default carried by `weight_bits` /
`act_bits`, keyed by layer name or by module, warning on a key the quantizer does not round:

```python
FakeQuantizer(model, weight_bits=8, layer_bits={'layer4.1.conv2': 4, 'fc': None})
```

In [ ]:
show_doc(FakeQuantizer.calibrate)

In [ ]:
show_doc(FakeQuantizer.quantize_model)

In [ ]:
show_doc(FakeQuantizer.remove)

In [ ]:
show_doc(FakeQuantizer.print_precision)

---

## Usage

Static activations, the ordinary case — measure the ranges, then round:

```python
from fasterai.quantize.fake_quantizer import FakeQuantizer

fq = FakeQuantizer(model, weight_bits=8, act_bits=8)
fq.calibrate(dls)          # observe the activation ranges (a `DataLoaders`, or `dls.train`)
fq.quantize_model()        # round in place, and return the same model
learn.validate()           # what W8A8 costs on your metric
fq.remove()                # bit-identical floating-point weights back
```

Weight-only, with no calibration to do. `group_size` has to divide every row it rounds, and a
convolution's row is `in_channels * kH * kW` — a ResNet's first convolution is 147 weights wide, which
no power of two divides, so name a size that fits or leave that layer out:

```python
FakeQuantizer(model, weight_bits=4, qscheme='per_group', group_size=64,
              layer_bits={'conv1': None}).quantize_model()
```

Dynamic activations, when you would rather not carry calibration data:

```python
FakeQuantizer(model, 8, 8, observer='dynamic').quantize_model()
```

`print_precision()` lists the width every layer was rounded to. It reports widths and nothing else: a
rounded model holds a floating-point copy of every weight it rounds, so it takes *more* memory while
rounded, not less, and runs a little slower because of the extra rounding. Those copies are
non-persistent buffers — the state dict keeps exactly the keys it had — and `remove()` frees them.

---

## See Also

- [Quantizer](quantizer.html) - Backend quantization that produces a model running at reduced precision
- [Precision](../core/precision.html) - `FakeQuantSpec`, and the precisions fasterai's backends can run
- [BN_Folder](../misc/bn_folding.html) - Fold BatchNorm before rounding, as a deployed flow would
- [Sparsifier](../sparse/sparsifier.html) - Zeroing weights, which survives this rounding
- [QuantizeCallback](quantize_callback.html) - Quantization inside a fastai training loop

Tests live in `nbs/tests/test_fake_quantizer.ipynb`.